# Project 6 — Push Pipeline with Generator Coroutines

This notebook rewrites the original pull pipeline as a **push-based coroutine pipeline**:

```text
cars.csv → source → filter 1 → filter 2 → ... → CSV sink
```

The implementation is intentionally reusable:

- any number of filters can be chained;
- filtering logic is represented by ordinary predicate functions;
- CSV parsing and writing use Python's `csv` module;
- downstream coroutines are closed automatically;
- matching is case-insensitive by default;
- output rows are preserved as strings, so values such as `4165.` are not reformatted.

## 1. Imports and coroutine priming helper

A coroutine must be advanced to its first `yield` before `.send(...)` can be used.
The decorator below performs that initialization once, at construction time.

In [17]:
import csv
from functools import wraps
from pathlib import Path


def coroutine(generator_function):
    """Create and automatically prime a generator-based coroutine."""
    @wraps(generator_function)
    def start(*args, **kwargs):
        generator = generator_function(*args, **kwargs)
        next(generator)
        return generator

    return start

## 2. Sink: write received rows to a CSV file

The sink is the final stage. It owns the output file and closes it when the
pipeline is closed. `newline=""` is important when using the `csv` module,
especially on Windows.

In [18]:
@coroutine
def csv_sink(
    output_path,
    include_header=False,
    header=None,
    encoding="utf-8",
):
    """Receive row sequences and write them to *output_path* as CSV.

    Parameters
    ----------
    output_path:
        Destination file path.
    include_header:
        When True, write *header* before any data rows.
    header:
        Optional sequence of column names.
    encoding:
        Text encoding used for the output file.
    """
    output_path = Path(output_path)

    if include_header and header is None:
        raise ValueError("header must be provided when include_header=True")

    # Create the parent directory when the caller uses a nested output path.
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with output_path.open("w", newline="", encoding=encoding) as output_file:
        writer = csv.writer(output_file, lineterminator="\n")

        if include_header:
            writer.writerow(header)

        try:
            while True:
                row = yield
                writer.writerow(row)
        except GeneratorExit:
            # Exiting the with-block flushes and closes the file.
            return

## 3. Generic filtering stage and predicate factory

`filter_rows` does not know anything about cars or names. It forwards a row only
when its predicate returns `True`.

`contains_text` creates a predicate for a specific field. Using `str.casefold()`
provides robust case-insensitive matching.

In [19]:
@coroutine
def filter_rows(predicate, target):
    """Forward rows satisfying *predicate* to the downstream *target*."""
    if not callable(predicate):
        raise TypeError("predicate must be callable")

    try:
        while True:
            row = yield
            if predicate(row):
                target.send(row)
    except GeneratorExit:
        # Closing one stage closes the rest of the pipeline.
        target.close()
        return


def contains_text(fragment, field_index=0, case_sensitive=False):
    """Return a predicate checking whether *fragment* occurs in one field."""
    if not isinstance(fragment, str):
        raise TypeError("fragment must be a string")
    if fragment == "":
        raise ValueError("fragment must not be empty")
    if field_index < 0:
        raise ValueError("field_index must be non-negative")

    needle = fragment if case_sensitive else fragment.casefold()

    def predicate(row):
        if field_index >= len(row):
            return False

        value = row[field_index]
        haystack = value if case_sensitive else value.casefold()
        return needle in haystack

    return predicate

## 4. Pipeline builder

The builder starts with the sink and wraps it in one filter stage per search
term. A row reaches the sink only if it passes **every** filter.

An empty `fragments` iterable is valid and produces a pipeline that writes every
non-empty input row.

In [20]:
def build_name_filter_pipeline(
    output_path,
    fragments,
    name_field_index=0,
    case_sensitive=False,
    include_header=False,
    header=None,
    encoding="utf-8",
):
    """Build a coroutine pipeline requiring all fragments in the name field."""
    fragments = tuple(fragments)

    target = csv_sink(
        output_path=output_path,
        include_header=include_header,
        header=header,
        encoding=encoding,
    )

    # Build from right to left so data flows through the filters in the same
    # order in which the fragments were supplied.
    for fragment in reversed(fragments):
        predicate = contains_text(
            fragment=fragment,
            field_index=name_field_index,
            case_sensitive=case_sensitive,
        )
        target = filter_rows(predicate, target)

    return target

## 5. Source: push rows from the input file

The source is a regular function rather than a coroutine. It reads rows and
pushes each one into the head of the pipeline. The `finally` block guarantees
that the complete pipeline is closed even when reading or processing fails.

In [21]:
def detect_csv_dialect(file_object, delimiters=",;\t|"):
    """Detect the CSV dialect without consuming the input stream.

    Falls back to a simple delimiter count when ``csv.Sniffer`` cannot infer
    a dialect from a small or irregular sample.
    """
    sample = file_object.read(8192)
    file_object.seek(0)

    if not sample:
        return csv.excel

    try:
        return csv.Sniffer().sniff(sample, delimiters=delimiters)
    except csv.Error:
        first_non_empty_line = next(
            (line for line in sample.splitlines() if line.strip()),
            "",
        )
        delimiter = max(
            delimiters,
            key=first_non_empty_line.count,
        )

        class FallbackDialect(csv.excel):
            pass

        FallbackDialect.delimiter = delimiter
        return FallbackDialect


In [22]:
def push_csv(
    source_path,
    target,
    skip_header=False,
    skip_blank_rows=True,
    strip_fields=True,
    input_delimiter=None,
    encoding="utf-8",
):
    """Read *source_path* and push parsed rows into *target*.

    Parameters
    ----------
    input_delimiter:
        Explicit input delimiter, such as `","` or `";"`. When omitted, the
        delimiter is detected automatically from common CSV delimiters.

    Returns
    -------
    int
        Number of non-blank data rows sent into the pipeline.
    """
    source_path = Path(source_path)
    rows_sent = 0

    try:
        with source_path.open("r", newline="", encoding=encoding) as source_file:
            if input_delimiter is None:
                dialect = detect_csv_dialect(source_file)
                reader = csv.reader(
                    source_file,
                    dialect=dialect,
                    skipinitialspace=True,
                )
            else:
                if not isinstance(input_delimiter, str) or len(input_delimiter) != 1:
                    raise ValueError(
                        "input_delimiter must be a single character or None"
                    )

                reader = csv.reader(
                    source_file,
                    delimiter=input_delimiter,
                    skipinitialspace=True,
                )

            if skip_header:
                next(reader, None)

            for row in reader:
                if strip_fields:
                    row = [field.strip() for field in row]

                if skip_blank_rows and not any(field for field in row):
                    continue

                target.send(row)
                rows_sent += 1
    finally:
        target.close()

    return rows_sent

## 6. High-level runner

This convenience function keeps notebook usage concise while leaving all
individual pipeline components available for reuse.

In [23]:
def run_name_filter_pipeline(
    source_path,
    output_path,
    fragments,
    name_field_index=0,
    case_sensitive=False,
    skip_header=False,
    strip_fields=True,
    input_delimiter=None,
    include_header=False,
    header=None,
    encoding="utf-8",
):
    """Build and execute the complete CSV name-filtering pipeline."""
    source_path = Path(source_path)
    output_path = Path(output_path)
    fragments = tuple(fragments)

    if not source_path.is_file():
        raise FileNotFoundError("Source CSV not found: {}".format(source_path))

    # Opening the sink primes it and therefore opens the output file immediately.
    # Prevent accidental truncation of the source file.
    if source_path.resolve() == output_path.resolve():
        raise ValueError("source_path and output_path must be different files")

    pipeline = build_name_filter_pipeline(
        output_path=output_path,
        fragments=fragments,
        name_field_index=name_field_index,
        case_sensitive=case_sensitive,
        include_header=include_header,
        header=header,
        encoding=encoding,
    )

    rows_read = push_csv(
        source_path=source_path,
        target=pipeline,
        skip_header=skip_header,
        strip_fields=strip_fields,
        input_delimiter=input_delimiter,
        encoding=encoding,
    )

    return {
        "source": str(source_path),
        "output": str(output_path),
        "filters": fragments,
        "rows_read": rows_read,
    }

## 7. Project example

All three fragments must occur in the name column. The default name field is
column `0`, and matching is case-insensitive.

In [24]:
SOURCE_FILE = Path("cars.csv")
OUTPUT_FILE = Path("chevrolet_monte_carlo_landau.csv")
NAME_FILTERS = ("Chevrolet", "Carlo", "Landau")

result = run_name_filter_pipeline(
    source_path=SOURCE_FILE,
    output_path=OUTPUT_FILE,
    fragments=NAME_FILTERS,

    # Leave this as None to auto-detect comma, semicolon, tab, or pipe.
    # For this particular file, input_delimiter=";" is also valid.
    input_delimiter=None,
)

result

{'source': 'cars.csv',
 'output': 'chevrolet_monte_carlo_landau.csv',
 'filters': ('Chevrolet', 'Carlo', 'Landau'),
 'rows_read': 407}

## 8. Verify the result

The project data should produce exactly two rows, both named
`Chevrolet Monte Carlo Landau`.

In [25]:
with OUTPUT_FILE.open("r", newline="", encoding="utf-8") as output_file:
    output_rows = list(csv.reader(output_file))

expected_name = "Chevrolet Monte Carlo Landau"
actual_names = [row[0] if row else None for row in output_rows]

assert len(output_rows) == 2, (
    "Expected 2 matching rows, found {}. Rows: {!r}".format(
        len(output_rows),
        output_rows,
    )
)
assert all(len(row) == 9 for row in output_rows), (
    "Expected 9 columns per row, found column counts {!r}. "
    "This usually indicates an incorrect input delimiter.".format(
        [len(row) for row in output_rows]
    )
)
assert all(name == expected_name for name in actual_names), (
    "Unexpected vehicle name(s): {!r}. Expected {!r}.".format(
        actual_names,
        expected_name,
    )
)

print(OUTPUT_FILE.read_text(encoding="utf-8"))

Chevrolet Monte Carlo Landau,15.5,8,350.0,170.0,4165.,11.4,77,US
Chevrolet Monte Carlo Landau,19.2,8,305.0,145.0,3425.,13.2,78,US



## Expected output

```text
Chevrolet Monte Carlo Landau,15.5,8,350.0,170.0,4165.,11.4,77,US
Chevrolet Monte Carlo Landau,19.2,8,305.0,145.0,3425.,13.2,78,US
```

### Reusing the pipeline

Change `NAME_FILTERS` to any number of fragments:

```python
NAME_FILTERS = ("Ford", "Mustang")
```

Use an empty tuple to copy every non-empty row:

```python
NAME_FILTERS = ()
```